# Function to automatic merge and fill missing values



## Create the function
You have to create a function which take 2 arguments :
- argument 1 : DataFrame main
- argument 2 : DataFrame opinion

Your function will return a new DataFrame which is the left merge of both DataFrames, and which has no missing values (so your function has to fill them), add new columns (flag, MONTH and DAY), and OPINION should be translated.

So you have to copy each useful code from the previous quests, and integrate it into a function. **This should achieve the same results as the previous quests.**

In [2]:
import pandas as pd

In [3]:
# Chargement des Datasets

link_main = "https://raw.githubusercontent.com/murpi/wilddata/master/quests/weather_main_2018.csv"
link_opinion = "https://raw.githubusercontent.com/murpi/wilddata/master/quests/weather_opinion_2018.csv"
df_main = pd.read_csv(link_main)
df_opinion = pd.read_csv(link_opinion)


In [4]:
# Affichage des premières lignes de df_main
df_main.head()


,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,PRESSURE_MAX_MB,CLOUDCOVER_AVG_PERCENT,HEATINDEX_MAX_C,DEWPOINT_MAX_C,WINDTEMP_MAX_C
0,2018-01-01,12,8,61,9,11,8,8.9,79,9.500,1018,41.750,12,8,7
1,2018-01-02,13,6,26,8,12,13,0.6,96,9.000,1020,87.875,13,12,6
2,2018-01-03,15,10,40,11,12,10,5.5,82,8.500,1017,91.500,15,13,7
3,2018-01-04,14,11,45,14,14,11,0.0,89,10.000,1011,90.125,14,12,10
4,2018-01-05,12,7,21,10,11,8,1.5,85,9.875,1005,62.375,12,10,7


In [5]:
# Affichage des premières lignes de df_opinion
df_opinion.head()

,date,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION
0,2018-01-01,113,0,3,5.1,bad
1,2018-03-12,119,0,2,8.8,bad
2,2018-03-09,116,0,3,10.2,bad
3,2018-10-07,122,0,1,5.6,bad
4,2018-06-18,119,0,1,12.9,bad


In [6]:
# Affichage du nombre de lignes et de colonnes de df_main
df_main.shape

(365, 15)

In [7]:
# Affichage des informations de df_main
df_main.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 15 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   DATE                    365 non-null    object 
 1   MAX_TEMPERATURE_C       365 non-null    int64  
 2   MIN_TEMPERATURE_C       365 non-null    int64  
 3   WINDSPEED_MAX_KMH       365 non-null    int64  
 4   TEMPERATURE_MORNING_C   365 non-null    int64  
 5   TEMPERATURE_NOON_C      365 non-null    int64  
 6   TEMPERATURE_EVENING_C   365 non-null    int64  
 7   PRECIP_TOTAL_DAY_MM     365 non-null    float64
 8   HUMIDITY_MAX_PERCENT    365 non-null    int64  
 9   VISIBILITY_AVG_KM       365 non-null    float64
 10  PRESSURE_MAX_MB         365 non-null    int64  
 11  CLOUDCOVER_AVG_PERCENT  365 non-null    float64
 12  HEATINDEX_MAX_C         365 non-null    int64  
 13  DEWPOINT_MAX_C          365 non-null    int64  
 14  WINDTEMP_MAX_C          365 non-null    in

In [8]:
# Nombre de valeurs nulles de df_main
df_main.isnull().sum()

DATE                      0
MAX_TEMPERATURE_C         0
MIN_TEMPERATURE_C         0
WINDSPEED_MAX_KMH         0
TEMPERATURE_MORNING_C     0
TEMPERATURE_NOON_C        0
TEMPERATURE_EVENING_C     0
PRECIP_TOTAL_DAY_MM       0
HUMIDITY_MAX_PERCENT      0
VISIBILITY_AVG_KM         0
PRESSURE_MAX_MB           0
CLOUDCOVER_AVG_PERCENT    0
HEATINDEX_MAX_C           0
DEWPOINT_MAX_C            0
WINDTEMP_MAX_C            0
dtype: int64

In [9]:
# Nombre de lignes et de colonnes de df_opinion
df_opinion.shape

(341, 6)

In [10]:
# Nombre de valeurs nulles de df_opinion
df_opinion.isnull().sum()

date                    0
WEATHER_CODE_EVENING    0
TOTAL_SNOW_MM           0
UV_INDEX                0
SUNHOUR                 0
OPINION                 0
dtype: int64

In [11]:
# Création d'une fonction merge_and_fill qui prend en paramètre
# deux DataFrame pour les joindre avec un left join sur la colonne de date

from datetime import datetime

def merge_and_fill(DataFrameMain, DataFrameOpinion):
  # MERGE BOTH DATAFRAMES
  NewDataFrame = pd.merge(left =DataFrameMain, right=DataFrameOpinion, left_on="DATE", right_on="date", how="left")

  # FILL IN MISSING VALUES
  # WEATHER_CODE_EVENING => mode
  NewDataFrame["WEATHER_CODE_EVENING"] = NewDataFrame["WEATHER_CODE_EVENING"].fillna(NewDataFrame["WEATHER_CODE_EVENING"].mode()[0])

  # TOTAL_SNOW_MM => moyenne
  NewDataFrame["TOTAL_SNOW_MM"] = NewDataFrame["TOTAL_SNOW_MM"].fillna(NewDataFrame["TOTAL_SNOW_MM"].mean())
  # UV_INDEX => médiane
  NewDataFrame["UV_INDEX"] = NewDataFrame["UV_INDEX"].fillna(NewDataFrame["TOTAL_SNOW_MM"].median())
  # SUNHOUR => bfill

  # OPINION => remplacer les nuls par "Unknown"
  NewDataFrame["OPINION"] = NewDataFrame["OPINION"].fillna("Unknown")
  # Ajout de la colonne FLAG pour OPINION nulles
  NewDataFrame["FLAG"] = NewDataFrame["OPINION"].isna()
  # Conversion de la colonne DATE au format Datetime
  NewDataFrame["DATE"] = pd.to_datetime(NewDataFrame["DATE"])
  # Création de la colonne MONTH
  NewDataFrame["MONTH"] = NewDataFrame["DATE"].dt.month
  # Création de la colonne DAY
  NewDataFrame["DAY"] = NewDataFrame["DATE"].dt.day
  # Tradudction de la colonne OPINION en français (créer une fonction)
  def translate(opinion):
    traduction = {"very bad": "très mauvais",
      "bad": "mauvais",
      "not good not bad": "ni bon ni mauvais",
      "good": "bon",
      "very good": "très bon",
    }
    opinion_c = opinion.strip().lower()
    return traduction.get(opinion_c, opinion)

  NewDataFrame["OPINION"] = NewDataFrame["OPINION"].apply(translate)
  # Return new merged dataframe
  return NewDataFrame

In [12]:
df_2018 = merge_and_fill(df_main, df_opinion)

In [13]:
# Afficher le nombre de lignes et de colonnes de df_2018
# Afficher les premières ignes de df_2018
df_2018.shape

(365, 24)

In [14]:
df_2018.head(5)


,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,WINDTEMP_MAX_C,date,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION,FLAG,MONTH,DAY
0,2018-01-01,12,8,61,9,11,8,8.9,79,9.500,...,7,2018-01-01,113.0,0.0,3.0,5.1,mauvais,False,1,1
1,2018-01-02,13,6,26,8,12,13,0.6,96,9.000,...,6,2018-01-02,122.0,0.0,3.0,3.3,mauvais,False,1,2
2,2018-01-03,15,10,40,11,12,10,5.5,82,8.500,...,7,2018-01-03,122.0,0.0,3.0,3.3,mauvais,False,1,3
3,2018-01-04,14,11,45,14,14,11,0.0,89,10.000,...,10,2018-01-04,116.0,0.0,3.0,3.3,mauvais,False,1,4
4,2018-01-05,12,7,21,10,11,8,1.5,85,9.875,...,7,2018-01-05,116.0,0.0,3.0,6.9,mauvais,False,1,5


In [15]:
# Afficher les informations de df_2018
df_2018.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 365 entries, 0 to 364
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   DATE                    365 non-null    datetime64[ns]
 1   MAX_TEMPERATURE_C       365 non-null    int64         
 2   MIN_TEMPERATURE_C       365 non-null    int64         
 3   WINDSPEED_MAX_KMH       365 non-null    int64         
 4   TEMPERATURE_MORNING_C   365 non-null    int64         
 5   TEMPERATURE_NOON_C      365 non-null    int64         
 6   TEMPERATURE_EVENING_C   365 non-null    int64         
 7   PRECIP_TOTAL_DAY_MM     365 non-null    float64       
 8   HUMIDITY_MAX_PERCENT    365 non-null    int64         
 9   VISIBILITY_AVG_KM       365 non-null    float64       
 10  PRESSURE_MAX_MB         365 non-null    int64         
 11  CLOUDCOVER_AVG_PERCENT  365 non-null    float64       
 12  HEATINDEX_MAX_C         365 non-null    int64     

## Execution with a loop
You will execute your function in a 9 rounds loop, once per year, and agregate this in a new big DataFrame with 24 columns and lots of rows.

In [16]:
# Chaque année entre 2010 et 2018 (inclus) a deux 2 sources : weather_main_YYYY.csv and weather_opinion_YYYY.csv
# Charger les deux datasets et joingez-les grâce à la fonction merge_and_fill
# Concaténez ce nouveau DataFrame au(x) précédent(s) pour n'obtenir qu'un long DataFrame df_weather

df_weather = pd.DataFrame()

# Créez la boucle :
for var_year in range(2010, 2019):
    # Collect both DataFrames each year
    year = str(var_year)
    link_main = "https://raw.githubusercontent.com/WildCodeSchool/data-training-resources/main/quests/weather_main_" + year + ".csv"
    link_opinion = "https://raw.githubusercontent.com/WildCodeSchool/data-training-resources/main/quests/weather_opinion_" + year + ".csv"
    df_main = pd.read_csv(link_main)
    df_opinion = pd.read_csv(link_opinion)

    # Apply your merge, cleaning and fillna function on the current year
    df_year = merge_and_fill(df_main , df_opinion)

    # Concat with the global DF
    df_weather = pd.concat([df_weather , df_year])


In [17]:
# Affichage des premières lignes de df_weather
df_weather.head(5)

,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,WINDTEMP_MAX_C,date,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION,FLAG,MONTH,DAY
0,2010-01-01,5,2,19,2,5,3,0.0,86,10.000,...,-1,2010-01-01,119.0,0.0,1.0,5.1,très mauvais,False,1,1
1,2010-01-02,5,-1,6,-1,4,1,0.0,78,10.000,...,2,2010-01-02,116.0,0.0,1.0,8.7,très mauvais,False,1,2
2,2010-01-03,5,1,26,2,4,2,0.3,61,9.875,...,-3,2010-01-03,119.0,0.0,1.0,6.9,très mauvais,False,1,3
3,2010-01-04,1,-4,25,-2,0,-3,0.0,75,10.000,...,-9,2010-01-04,113.0,0.0,1.0,8.7,très mauvais,False,1,4
4,2010-01-05,2,-5,20,-5,1,-4,0.0,71,10.000,...,-10,2010-01-05,113.0,0.0,1.0,8.7,très mauvais,False,1,5


In [18]:
# Création d'une fonction clean qui :
# * crée une colonne FLAG indiquant une valeur manquante
# * convertit la colonne DATE au format datetime
# * traduit l'opinion en français

def clean(DataFrameYear):

  # ALIGN DATAFRAME SHAPE, DATATYPE

  # Colonne FLAG
  DataFrameYear["FLAG"] = DataFrameYear["OPINION"].isna()

  # Colone DATE
  DataFrameYear["DATE"] = pd.to_datetime(DataFrameYear["DATE"])

  # Colonne OPINION
  def translate(opinion):
    traduction = {"very bad": "très mauvais",
      "bad": "mauvais",
      "not good not bad": "ni bon ni mauvais",
      "good": "bon",
      "very good": "très bon",
    }
    opinion_c = opinion.strip().lower()
    return traduction.get(opinion_c, opinion)

  DataFrameYear["OPINION"] = DataFrameYear["OPINION"].apply(translate)
  # Return new cleaned dataframe
  return DataFrameYear

In [19]:
# On crée une liste de DataFrames pour les années 2019 à 2022 (inclus)

links = {}
for year in range(2019, 2023):
    # Collect both DataFrames each year
    year = str(year)
    links[year] = "https://raw.githubusercontent.com/WildCodeSchool/data-training-resources/main/quests/weather" + year + ".csv"

# Create dataframes from the links to clean
dfs = {}  # Dictionary to store dataframes
for year, link in links.items():
    dfs[year] = pd.read_csv(link)
    dfs[year] = clean(dfs[year])

dfs['2019']

,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,WEATHER_CODE_MORNING,WEATHER_CODE_NOON,WEATHER_CODE_EVENING,TOTAL_SNOW_MM,UV_INDEX,SUNHOUR,OPINION,MONTH,DAY,FLAG
0,2019-01-01,9,4,10,4,7,8,0.2,94,9.000,...,116,143,176,0,1,5.1,très mauvais,1,1,False
1,2019-01-02,8,5,18,7,7,5,0.0,90,9.000,...,119,116,116,0,1,8.7,très mauvais,1,2,False
2,2019-01-03,6,0,18,0,4,3,0.0,88,10.000,...,116,116,116,0,1,8.7,très mauvais,1,3,False
3,2019-01-04,5,-1,15,-1,4,3,0.0,91,10.000,...,116,116,122,0,1,5.1,très mauvais,1,4,False
4,2019-01-05,6,-1,8,-1,4,3,0.0,91,8.000,...,143,116,116,0,1,8.7,très mauvais,1,5,False
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
360,2019-12-27,13,10,13,10,12,11,0.0,96,7.375,...,119,143,116,0,1,8.7,ni bon ni mauvais,12,27,False
361,2019-12-28,11,5,12,7,8,8,0.0,93,7.625,...,143,116,116,0,1,6.9,très mauvais,12,28,False
362,2019-12-29,9,2,11,3,6,7,0.0,86,10.000,...,116,116,116,0,1,8.7,très mauvais,12,29,False
363,2019-12-30,12,4,7,5,9,9,0.0,61,10.000,...,113,113,113,0,1,8.7,très mauvais,12,30,False


In [20]:
# Concaténez df_weather avec df_2019 à 2022
df_weather = pd.concat([df_weather] + list(dfs.values()), ignore_index=True)

In [21]:
# Affichez les premières lignes de df_weather
df_weather.head(5)

,DATE,MAX_TEMPERATURE_C,MIN_TEMPERATURE_C,WINDSPEED_MAX_KMH,TEMPERATURE_MORNING_C,TEMPERATURE_NOON_C,TEMPERATURE_EVENING_C,PRECIP_TOTAL_DAY_MM,HUMIDITY_MAX_PERCENT,VISIBILITY_AVG_KM,...,SUNHOUR,OPINION,FLAG,MONTH,DAY,WEATHER_CODE_MORNING,WEATHER_CODE_NOON,SUNSET,SUNRISE,TEMPERATURE_NIGHT_C
0,2010-01-01,5,2,19,2,5,3,0.0,86,10.000,...,5.1,très mauvais,False,1.0,1.0,NaN,NaN,NaN,NaN,NaN
1,2010-01-02,5,-1,6,-1,4,1,0.0,78,10.000,...,8.7,très mauvais,False,1.0,2.0,NaN,NaN,NaN,NaN,NaN
2,2010-01-03,5,1,26,2,4,2,0.3,61,9.875,...,6.9,très mauvais,False,1.0,3.0,NaN,NaN,NaN,NaN,NaN
3,2010-01-04,1,-4,25,-2,0,-3,0.0,75,10.000,...,8.7,très mauvais,False,1.0,4.0,NaN,NaN,NaN,NaN,NaN
4,2010-01-05,2,-5,20,-5,1,-4,0.0,71,10.000,...,8.7,très mauvais,False,1.0,5.0,NaN,NaN,NaN,NaN,NaN


# Draw a scatterplot
With Date on X-Axis and Max Temperature on Y-axis

The result must have 11 seasonality for the 11 summers/winters alternance.

Utilisez Seaborn ou Plotly Express selon votre préférence




In [22]:
%pip install plotly

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [23]:
%pip install nbformat

Defaulting to user installation because normal site-packages is not writeable
Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0 -> 26.2.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [24]:
import matplotlib.pyplot as plt
import plotly.express as px
fig = px.scatter(
    df_weather,
    x="DATE",
    y="MAX_TEMPERATURE_C",color="MAX_TEMPERATURE_C", color_continuous_scale=px.colors.sequential.Jet,title= "MAX TEMPERATURES EVOLUTION", labels={"DATE": "Date", "MAX_TEMPERATURE_C": "Maximum temperatures (Celsius degrees)"})
fig.show()


Find appropriate dataviz to answer to this question :

Which year had the best weather? Are years very different or not ?


In [25]:
df_weather["YEAR"] = pd.to_datetime(df_weather["DATE"]).dt.year
df_filtre = df_weather[df_weather["OPINION"].isin(["bon", "très bon", "météo favorable","météo idéale"])]
df_aggré = df_filtre.groupby("YEAR").size().reset_index(name="Avis")
df_aggré["YEAR_STR"] = df_aggré["YEAR"].astype(str)

fig = px.bar(
    df_aggré,
    x="YEAR",
    y="Avis",
    color="YEAR_STR",
    color_discrete_sequence=px.colors.qualitative.Set2,
    title="Nombre d'opinions Bon / Très bon par année",
    labels={"YEAR": "Année", "YEAR_STR": "Année"})
bargap=0.2
fig.show()

Quelles années semblent avoir bénéficié des meilleures conditions météo ? Pour quelles raisons ?

In [ ]:
#Très certainement 2022 en ce basant sur le nombre d'avis de météo favorable, car "une bonne météo" est forcément subjective, c'est donc a quel point est agréable.